# Phase 2A — acquire the original SWAN-SF archives

This notebook only acquires and verifies the seven original [Harvard Dataverse SWAN-SF](https://doi.org/10.7910/DVN/EBCFKM) archives (released dataset version 1.2). It does **not** extract data, preprocess, or train models.

**Safe default:** Run All may fetch metadata and ask you to mount/authenticate Drive, but the final acquisition cell is disabled until you edit its explicit confirmation string. Review the live metadata table and measured storage before enabling it. The archive files and manifest stay in `MyDrive/Solar-Flare-Project/datasets/raw/`, outside Git.

## 1. Runtime diagnostics

Run in a real Google Colab runtime. These values are measured here, not assumed in advance.

In [ ]:
import os
import platform
import shutil
import sys
from pathlib import Path

def gib(n):
    return f"{n / (1024 ** 3):.2f} GiB"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Logical CPUs:", os.cpu_count())
meminfo = {}
with open("/proc/meminfo", encoding="utf-8") as handle:
    for line in handle:
        key, value = line.split(":", 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
print("RAM total:", gib(meminfo["MemTotal"]))
print("RAM available:", gib(meminfo["MemAvailable"]))
print("/content free:", gib(shutil.disk_usage("/content").free))

## 2. Mount Google Drive — user action

Run this cell and grant access to **your own** Drive. Mounting is separate from the Drive API authorization used for a measured storage-quota check. Do not paste credentials into the notebook.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Select persistent storage and revalidate Dataverse metadata

The table below contains the metadata verified during notebook preparation. This cell fetches the **live official dataset record** and requires an exact match on released version, filename, file ID, byte size, and MD5 before any acquisition can be enabled. If the published record changes, stop and review it; do not silently substitute files.

Official API: https://dataverse.harvard.edu/api/datasets/:persistentId/?persistentId=doi%3A10.7910%2FDVN%2FEBCFKM

In [ ]:
import json
from datetime import datetime, timezone
from urllib.parse import quote
from urllib.request import Request, urlopen

DOI = "doi:10.7910/DVN/EBCFKM"
METADATA_URL = (
    "https://dataverse.harvard.edu/api/datasets/:persistentId/"
    "?persistentId=" + quote(DOI, safe="")
)
RAW_DIR = Path("/content/drive/MyDrive/Solar-Flare-Project/datasets/raw")
EXPECTED = {
    "SWAN.tar.gz": (3642432, 735206766, "379da8ba2920a82c81d1c4d3293fc709"),
    "addenda.tar.gz": (3642452, 34042374, "82c4ab7d6593cb34dfb4d9c216767eae"),
    "partition1_instances.tar.gz": (3642446, 1273859189, "b54602205c0a0d997501043ad8364e4f"),
    "partition2_instances.tar.gz": (3642449, 1512325420, "6122eaa088c2ae75551c6f12efa3d081"),
    "partition3_instances.tar.gz": (3642433, 736247805, "4b291532dc5d17b327807957c314f03f"),
    "partition4_instances.tar.gz": (3642451, 885390803, "f0cee52b3e5346eaaa5a2fc6bc05bd64"),
    "partition5_instances.tar.gz": (3642450, 1301114492, "425e58374798e4c34cc99375c0d2bc27"),
}

def official_metadata():
    request = Request(METADATA_URL, headers={"User-Agent": "SWAN-SF-phase2a/1"})
    with urlopen(request, timeout=30) as response:
        payload = json.load(response)
    if payload.get("status") != "OK":
        raise RuntimeError("Dataverse metadata API did not return status OK")
    version = payload["data"]["latestVersion"]
    observed_version = f"{version['versionNumber']}.{version['versionMinorNumber']}"
    if observed_version != "1.2" or version.get("versionState") != "RELEASED":
        raise RuntimeError(f"Expected released SWAN-SF version 1.2; got {observed_version} / {version.get('versionState')}")
    records = {}
    for item in version["files"]:
        name = item["label"]
        data_file = item["dataFile"]
        if name in records:
            raise RuntimeError(f"Duplicate Dataverse filename: {name}")
        records[name] = {
            "filename": name,
            "file_id": int(data_file["id"]),
            "expected_size": int(data_file["filesize"]),
            "checksum_type": data_file["checksum"]["type"].upper(),
            "published_md5": data_file["checksum"]["value"].lower(),
            "source_url": f"https://dataverse.harvard.edu/api/access/datafile/{data_file['id']}",
            "restricted": bool(item.get("restricted", False)),
        }
    if set(records) != set(EXPECTED):
        raise RuntimeError(f"Dataverse file set differs: {sorted(records)}")
    for name, expected in EXPECTED.items():
        rec = records[name]
        actual = (rec["file_id"], rec["expected_size"], rec["published_md5"])
        if actual != expected or rec["checksum_type"] != "MD5" or rec["restricted"]:
            raise RuntimeError(f"Official metadata mismatch or restricted file: {name}: {rec}")
    return records

OFFICIAL = official_metadata()
print("Official DOI:", DOI, "| released version: 1.2")
print("Persistent destination:", RAW_DIR)
for name, rec in OFFICIAL.items():
    print(name, "| ID", rec["file_id"], "| bytes", rec["expected_size"], "| MD5", rec["published_md5"])
print("Total archive bytes:", sum(r["expected_size"] for r in OFFICIAL.values()))

## 4. Storage preflight — user action

The first acquisition requires at least **8 GiB of measured Drive quota free**, plus at least **2 GiB free on /content** for runtime headroom. On recovery, previously verified archives count toward the original 8 GiB reservation; the remaining free quota must still cover every missing archive plus 256 MiB. The notebook uses the Google Drive API `storageQuota(limit,usage)`; a mounted Drive filesystem's disk-usage reading is **not** treated as your account quota. If the API cannot report a finite limit and usage, acquisition stops.

This cell authenticates for a read-only quota query. It verifies any existing final archive by byte size and streamed MD5 before counting it. It does not download or remove anything.

In [ ]:
import hashlib
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

GIB = 1024 ** 3
MIB = 1024 ** 2
MIN_DRIVE_FREE = 8 * GIB
MIN_CONTENT_FREE = 2 * GIB
TRANSFER_MARGIN = 256 * MIB
DRIVE_SERVICE = None

def hash_file(path):
    md5 = hashlib.md5()
    sha256 = hashlib.sha256()
    size = 0
    with Path(path).open("rb") as handle:
        while chunk := handle.read(8 * MIB):
            md5.update(chunk)
            sha256.update(chunk)
            size += len(chunk)
    return size, md5.hexdigest(), sha256.hexdigest()

def verified_final_bytes():
    total = 0
    for name, rec in OFFICIAL.items():
        path = RAW_DIR / name
        if path.exists():
            if not path.is_file():
                raise RuntimeError(f"Archive path is not a file: {path}")
            size, md5, _ = hash_file(path)
            if size != rec["expected_size"] or md5 != rec["published_md5"]:
                raise RuntimeError(f"Existing final archive failed verification; preserved: {path}")
            total += size
    return total

def drive_quota_free():
    if DRIVE_SERVICE is None:
        raise RuntimeError("Drive API authorization is missing")
    quota = DRIVE_SERVICE.about().get(fields="storageQuota(limit,usage)").execute().get("storageQuota", {})
    if quota.get("limit") is None or quota.get("usage") is None:
        raise RuntimeError("Drive API did not report finite quota and usage; cannot measure free space")
    free = int(quota["limit"]) - int(quota["usage"])
    if free < 0:
        raise RuntimeError("Drive account is over quota")
    return free

def storage_preflight():
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount your Google Drive first")
    if len(OFFICIAL) != 7:
        raise RuntimeError("Live Dataverse metadata has not been validated")
    drive_free = drive_quota_free()
    content_free = shutil.disk_usage("/content").free
    verified_bytes = verified_final_bytes()
    remaining = sum(r["expected_size"] for r in OFFICIAL.values()) - verified_bytes
    required_drive = max(MIN_DRIVE_FREE - verified_bytes, remaining + TRANSFER_MARGIN)
    print("Drive quota free:", gib(drive_free))
    print("/content free:", gib(content_free))
    print("Verified archive bytes already present:", verified_bytes)
    print("Archive bytes still needed:", remaining)
    print("Drive free required for this run:", gib(required_drive))
    if drive_free < required_drive or content_free < MIN_CONTENT_FREE:
        raise RuntimeError("Storage preflight failed; no download is allowed")
    return {"drive_free": drive_free, "content_free": content_free, "verified_bytes": verified_bytes}

auth.authenticate_user()
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.metadata.readonly"])
DRIVE_SERVICE = build("drive", "v3", credentials=credentials, cache_discovery=False)
PREFLIGHT = storage_preflight()
print("Storage preflight passed at", datetime.now(timezone.utc).isoformat())

## 5. Acquisition helpers

Defining these functions does **not** download archives. Downloads use the official file endpoints and write `.part` files under the persistent raw-data folder. A partial file is resumed only after the server responds with HTTP 206 and a matching `Content-Range`. When resume is unavailable, a new uniquely named temporary file is used; the old partial is preserved. Network retries are bounded. A file is promoted to its final name only after complete size and MD5 verification. The final name is never overwritten.

In [ ]:
import http.client
import os
import re
import time
import uuid
from urllib.error import HTTPError, URLError

MAX_ATTEMPTS = 3
CHUNK_BYTES = 8 * MIB

class RangeUnsupported(Exception):
    pass

def new_part(name):
    return RAW_DIR / f"{name}.{uuid.uuid4().hex}.part"

def stream_from_dataverse(rec, part):
    offset = part.stat().st_size if part.exists() else 0
    if offset > rec["expected_size"]:
        raise RuntimeError(f"Partial is larger than official size: {part}")
    headers = {"User-Agent": "SWAN-SF-phase2a/1"}
    if offset:
        headers["Range"] = f"bytes={offset}-"
    request = Request(rec["source_url"], headers=headers)
    with urlopen(request, timeout=60) as response:
        if offset:
            content_range = response.headers.get("Content-Range", "")
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
            if (response.status != 206 or match is None
                    or int(match.group(1)) != offset
                    or int(match.group(2)) != rec["expected_size"] - 1
                    or int(match.group(3)) != rec["expected_size"]):
                raise RangeUnsupported(f"Server did not confirm resume offset {offset}")
        elif response.status != 200:
            raise RuntimeError(f"Unexpected HTTP status {response.status}")
        transferred = offset
        next_report = ((offset // (64 * MIB)) + 1) * (64 * MIB)
        with part.open("ab") as output:
            while chunk := response.read(CHUNK_BYTES):
                output.write(chunk)
                transferred += len(chunk)
                if transferred > rec["expected_size"]:
                    raise RuntimeError(f"Received more bytes than published for {rec['filename']}")
                if transferred >= next_report:
                    print(f"  {rec['filename']}: {transferred:,}/{rec['expected_size']:,} bytes", flush=True)
                    next_report += 64 * MIB
            output.flush()
            os.fsync(output.fileno())
    print(f"  {rec['filename']}: {transferred:,}/{rec['expected_size']:,} bytes", flush=True)

def available_partial(name, expected_size):
    candidates = sorted(
        (path for path in RAW_DIR.glob(name + ".*.part") if path.is_file() and path.stat().st_size <= expected_size),
        key=lambda path: path.stat().st_size,
        reverse=True,
    )
    return candidates[0] if candidates else new_part(name)

def acquire_one(rec):
    final = RAW_DIR / rec["filename"]
    if final.exists():
        size, md5, sha256 = hash_file(final)
        if size != rec["expected_size"] or md5 != rec["published_md5"]:
            raise RuntimeError(f"Existing final file failed verification; preserved: {final}")
        print(f"SKIP verified: {rec['filename']}")
        return size, md5, sha256
    part = available_partial(rec["filename"], rec["expected_size"])
    for attempt in range(1, MAX_ATTEMPTS + 1):
        offset = part.stat().st_size if part.exists() else 0
        if drive_quota_free() < rec["expected_size"] - offset + TRANSFER_MARGIN:
            raise RuntimeError(f"Insufficient measured Drive quota for {rec['filename']}")
        print(f"Attempt {attempt}/{MAX_ATTEMPTS}: {rec['filename']} from byte {offset:,}")
        try:
            if offset < rec["expected_size"]:
                stream_from_dataverse(rec, part)
            size = part.stat().st_size
            if size < rec["expected_size"]:
                print(f"  Incomplete response ({size:,} bytes); partial preserved for retry")
                if attempt < MAX_ATTEMPTS:
                    time.sleep(min(2 ** attempt, 8))
                continue
            size, md5, sha256 = hash_file(part)
            if size != rec["expected_size"] or md5 != rec["published_md5"]:
                raise RuntimeError(
                    f"Size/MD5 mismatch for {part.name}; expected {rec['expected_size']:,} bytes / "
                    f"{rec['published_md5']}, got {size:,} bytes / {md5}. Partial preserved."
                )
            if final.exists():
                raise RuntimeError(f"Final name appeared during transfer; preserving both: {final}")
            part.rename(final)
            print(f"VERIFIED: {rec['filename']} | SHA-256 {sha256}")
            return size, md5, sha256
        except RangeUnsupported as exc:
            print(f"  Resume unavailable ({exc}); preserving {part.name} and starting a new temporary file")
            part = new_part(rec["filename"])
        except HTTPError as exc:
            if exc.code == 416 and offset:
                print(f"  Server rejected resume; preserving {part.name} and starting a new temporary file")
                part = new_part(rec["filename"])
                continue
            if exc.code not in (408, 429, 500, 502, 503, 504):
                raise
            print(f"  Transient HTTP {exc.code}; partial preserved")
        except (URLError, TimeoutError, ConnectionResetError, http.client.IncompleteRead) as exc:
            print(f"  Transient connection error: {type(exc).__name__}; partial preserved")
        if attempt < MAX_ATTEMPTS:
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"Download did not complete after {MAX_ATTEMPTS} attempts; partial files preserved")

def write_manifest(entries):
    payload = {
        "dataset_doi": DOI,
        "dataset_version": "1.2",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "files": entries,
    }
    target = RAW_DIR / "acquisition_manifest.json"
    temporary = RAW_DIR / f"acquisition_manifest.{uuid.uuid4().hex}.tmp"
    with temporary.open("x", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(target)

## 6. Manual acquisition — **disabled by default**

Run this cell only after reviewing the live metadata and actual storage measurements. To enable it, replace the blank confirmation value with the exact string shown in the cell. **Run All with the default value cannot download archives.**

The cell refreshes official metadata and storage preflight before any transfer. It records one manifest entry per file, updates the manifest after each attempt, and prints a final status table. If a run fails, leave the verified files and `.part` files in place for recovery.

In [ ]:
CONFIRM_ACQUISITION = ""  # Manually change to "DOWNLOAD_VERIFIED_SWAN_SF_V1_2" only when ready.

if CONFIRM_ACQUISITION != "DOWNLOAD_VERIFIED_SWAN_SF_V1_2":
    print("Acquisition disabled. No archive downloads were started.")
else:
    current = official_metadata()
    if current != OFFICIAL:
        raise RuntimeError("Official metadata changed since review; stop before downloading")
    storage_preflight()
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    entries = []
    for name, rec in current.items():
        entry = {
            "filename": name,
            "file_id": rec["file_id"],
            "source_url": rec["source_url"],
            "expected_size": rec["expected_size"],
            "published_checksum": {"algorithm": "MD5", "value": rec["published_md5"]},
            "actual_size": None,
            "sha256": None,
            "verified_at_utc": None,
            "verification_status": "pending",
        }
        try:
            size, md5, sha256 = acquire_one(rec)
            entry.update({
                "actual_size": size,
                "actual_md5": md5,
                "sha256": sha256,
                "verified_at_utc": datetime.now(timezone.utc).isoformat(),
                "verification_status": "verified",
            })
        except Exception as exc:
            entry["verification_status"] = "failed"
            present = [path for path in [RAW_DIR / name, *RAW_DIR.glob(name + ".*.part")] if path.is_file()]
            entry["actual_size"] = max((path.stat().st_size for path in present), default=0)
            entry["error_type"] = type(exc).__name__
            entry["error"] = str(exc)
            print(f"FAILED: {name}: {type(exc).__name__}: {exc}")
        entries.append(entry)
        write_manifest(entries)
    print("\nFINAL VERIFICATION REPORT")
    for entry in entries:
        print(f"{entry['filename']}: {entry['verification_status']} "
              f"(expected {entry['expected_size']:,} bytes; actual {entry['actual_size']})")
    print("Manifest:", RAW_DIR / "acquisition_manifest.json")
    print("Verified:", sum(e["verification_status"] == "verified" for e in entries), "of", len(entries))

## 7. Recovery and completion

After a runtime reset, reconnect, rerun the diagnostics, mount, live metadata, and storage-preflight cells. Then manually enable and run the acquisition cell again. It rehashes existing final archives before skipping them. It attempts to resume the largest eligible partial only if the server confirms the byte range; otherwise it starts a new temporary file and preserves the old one.

Inspect `acquisition_manifest.json` and the final report. Completion means **all seven** entries say `verified`, with expected byte sizes, published MD5 values, and recorded SHA-256 values. Failed or incomplete files remain in the persistent folder for review. This notebook does not extract archives or run preprocessing.

If Drive quota cannot be measured, is insufficient, or the Drive mount/API fails, stop and resolve that condition before rerunning acquisition. Avoid running two copies of this notebook concurrently against the same destination.